# Research Graph Tools: a gradual walkthrough

Run the cells from top to bottom. From the repository folder, run `uv sync`, then open this notebook with Jupyter or VS Code and select the environment created in `.venv`. The examples use small, connected, undirected, unweighted graphs.

## 1. Create a graph

A graph is a set of vertices joined by edges. A path connects vertices in a line; a cycle also connects the last vertex back to the first. The constructors return ordinary NetworkX graphs with integer vertex IDs.

In [ ]:
from research_graph_tools.graphs import path_graph, cycle_graph

path = path_graph(4)
cycle = cycle_graph(4)

## 2. Look at its parts

Use NetworkX to inspect the vertices and edges. An edge `(u, v)` means those two vertices are connected.

In [ ]:
print("Nodes:", list(path.nodes))
print("Edges:", list(path.edges))
print("Number of nodes:", path.number_of_nodes())
print("Number of edges:", path.number_of_edges())

## 3. Compare two simple graphs

The path and cycle have the same four vertices, but the cycle has one extra edge connecting its ends.

In [ ]:
print("Path edges:", list(path.edges))
print("Cycle edges:", list(cycle.edges))

## 4. Calculate one number

Kemeny’s constant is the expected number of steps to reach a destination chosen according to the random walk’s long-run visit probabilities. For a connected graph, this value is independent of the starting vertex. The function returns one number.

In [ ]:
from research_graph_tools.metrics import kemeny_constant

k = kemeny_constant(path)
print("Kemeny's constant:", k)

## 5. Get results for each node

Accessibility and the Fiedler vector return one value per vertex, as dictionaries keyed by vertex ID. Accessibility summarizes expected hitting times to each vertex; the Fiedler value gives that vertex’s coordinate along a Laplacian eigenvector.

In [ ]:
from research_graph_tools.metrics import accessibility_index, fiedler_vector

accessibility = accessibility_index(path)
fiedler = fiedler_vector(path)
print("Accessibility:", accessibility)
print("Fiedler vector:", fiedler)

## 6. Calculate an approximate conductance

Conductance measures how weakly a graph can be split into two parts. This function uses a sweep over the normalized Fiedler vector and returns the conductance of the best cut it finds; it is an approximation to the Cheeger constant.

In [ ]:
from research_graph_tools.metrics import cheeger_approximation

print("Approximate conductance:", cheeger_approximation(path))

## 7. Draw and compare graphs

`draw_graph(graph)` shows one graph. Hover over a vertex to inspect its metrics. Use the dropdown to choose which metric colors the vertices. Pass an added edge to compare the original graph with the augmented graph; `show_delta=True` adds a third panel showing metric changes.

In [ ]:
from IPython.display import HTML, display
from research_graph_tools.visualization import draw_graph

display(HTML(draw_graph(path).to_html()))
display(HTML(draw_graph(path, (0, 3), show_delta=True).to_html()))

## 8. Search candidates

`find_counterexample` checks candidates in order and returns the first one whose predicate is true. Here the candidates are paths of increasing size, and the predicate asks whether Kemeny’s constant exceeds 5. The limit bounds how many graphs are checked.

In [ ]:
from research_graph_tools.search import find_counterexample

candidates = (path_graph(n) for n in range(3, 10))
threshold = 5
found = find_counterexample(
    candidates,
    lambda graph: kemeny_constant(graph) > threshold,
    limit=7,
)
if found is None:
    print("No candidate passed the condition.")
else:
    print("First match:", found.number_of_nodes(), "vertices")
    print("Kemeny’s constant:", kemeny_constant(found))

## 9. Use your own graph

You can build a NetworkX graph directly and pass it to the same calculations and visualization tools. The metric functions require connected, undirected, unweighted graphs.

In [ ]:
import networkx as nx

custom = nx.Graph()
custom.add_edges_from([("A", "B"), ("B", "C"), ("C", "A"), ("C", "D")])
print("Kemeny's constant:", kemeny_constant(custom))
display(HTML(draw_graph(custom, color_metric="fiedler").to_html()))

## 10. Visualize a graph from an adjacency matrix

An adjacency matrix has one row and column per vertex; a `1` marks an edge and `0` means no edge. For an undirected graph the matrix is symmetric. Pass the NumPy array straight to `draw_graph`; it creates numbered vertices and chooses positions automatically.

In [ ]:
import numpy as np

# Rows and columns are vertices 0, 1, 2, and 3.
# A 1 means those two vertices share an edge.
adjacency = np.array([
    [0, 1, 0, 1],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [1, 0, 1, 0],
])

display(HTML(draw_graph(adjacency).to_html()))
# Compare the cycle with a new edge between vertices 0 and 2.
display(HTML(draw_graph(adjacency, (0, 2), show_delta=True).to_html()))